# Semantic Search vs. Filter Objects: When to Use Each
**EPS Research RAG Corpus Series**

This notebook compares `semantic_search` and `filter_objects` for the same scientific goal,
showing when each tool is the right choice.

**Rule of thumb:**
- Use `filter_objects` when you know the field name and want an exact numeric range
- Use `semantic_search` when you want discovery, similarity, or don't know the field name

In [1]:
import requests
import pandas as pd

BASE = 'https://dflynn5656-astro-rag-mcp.hf.space/api'

def semantic_search(corpus, query, top_k=10):
    r = requests.get(f'{BASE}/semantic_search', params={'corpus': corpus, 'query': query, 'top_k': top_k})
    return r.json()['data']

def filter_objects(corpus, field, min_val=None, max_val=None):
    params = {'corpus': corpus, 'field': field}
    if min_val is not None: params['min'] = min_val
    if max_val is not None: params['max'] = max_val
    r = requests.get(f'{BASE}/filter_objects', params=params)
    return r.json()['data']

print('Ready.')

Ready.


## Case 1: Finding nearby galaxies
**Goal:** Find galaxies within 10 Mpc in the v7 corpus.

Here `filter_objects` is clearly better — we know the field name and have an exact range.

In [2]:
# filter_objects — exact and fast
result_filter = filter_objects('v7', 'distance_mpc', min_val=1, max_val=10)
print(f'filter_objects: {result_filter["n_matched"]} galaxies between 1 and 10 Mpc')

# semantic_search — approximate, field-name-free
result_semantic = semantic_search('v7', 'nearby local universe galaxy within 10 Mpc', top_k=10)
print(f'semantic_search: {result_semantic["n_returned"]} results returned')
print('Top IDs:', [r['id'] for r in result_semantic['results']])

filter_objects: 119 galaxies between 1 and 10 Mpc


semantic_search: 10 results returned
Top IDs: ['NGC1090', 'NGC3992', 'NGC1003', 'NGC3949', 'UGC10310', 'F579-V1', 'NGC4010', 'NGC4559', 'WALLABY_J101035-254920', 'WALLABY_J103918-265030']


**Verdict:** For known-field numeric queries, use `filter_objects`. It returns all 119 qualifying objects; semantic search returns 10 approximate matches.

## Case 2: Finding galaxies 'like' a known object
**Goal:** Find galaxies similar to NGC3198 — a well-studied Sc spiral.

Here `semantic_search` wins — there's no single field that captures 'similar to NGC3198'.

In [3]:
# semantic_search — captures holistic similarity
result = semantic_search('v7', 'Sc spiral high rotation velocity tilted ring well-resolved', top_k=5)
print('Galaxies similar to NGC3198:')
for r in result['results']:
    print(f"  {r['id']:20s}  score={r['score']:.3f}")

Galaxies similar to NGC3198:
  DDO_46                score=1.134
  DDO_101               score=1.135
  IC_10                 score=1.135
  IC_1613               score=1.135
  DDO_53                score=1.138


## Case 3: Exploring without schema knowledge
**Goal:** Find compact high-redshift star-forming galaxies in intz — but you don't know the field names.

Use `semantic_search`; use `get_corpus_schema` to learn field names for follow-up filtering.

In [4]:
# Step 1: Semantic discovery
result = semantic_search('intz', 'compact high redshift star forming high SFR', top_k=5)
print('Top intz objects:')
for r in result['results']:
    print(f"  {r['id']:30s}  score={r['score']:.3f}")

# Step 2: Learn the schema
schema = requests.get(f'{BASE}/get_corpus_schema', params={'corpus': 'intz'}).json()
fields = list(schema['data'].get('fields', {}).keys())[:15]
print(f'\nAvailable fields (first 15): {fields}')

Top intz objects:
  C-HiZ_z1_265                    score=0.833
  C-HiZ_z1_267                    score=0.840
  C-HiZ_z1_285                    score=0.840
  C-HiZ_z1_268                    score=0.840
  C-HiZ_z1_334                    score=0.843



Available fields (first 15): []


## Summary

| Task | Use |
|------|-----|
| Exact numeric range (distance, redshift, velocity) | `filter_objects` |
| Object similarity / discovery | `semantic_search` |
| Unknown field names | `semantic_search` + `get_corpus_schema` |
| Cross-epoch comparison | `semantic_search` across corpora |
| Complete result set | `filter_objects` (returns all matches) |